# Week 05

In [1]:
from typing import List, Tuple


class DisjointSet:
    """Union-Find (Disjoint Set Union) with path compression + union by rank."""
    def __init__(self, n: int):
        self._parent = [i for i in range(n)]
        self._rank = [0 for _ in range(n)]

    def find(self, x: int) -> int:
        """Return representative of x's set."""
        if self._parent[x] != x:
            self._parent[x] = self.find(self._parent[x])
        return self._parent[x]

    def union(self, a: int, b: int) -> bool:
        """Union sets; return True if merged (i.e., were different)."""
        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return False

        # Union by rank
        if self._rank[ra] < self._rank[rb]:
            self._parent[ra] = rb
        elif self._rank[ra] > self._rank[rb]:
            self._parent[rb] = ra
        else:
            self._parent[rb] = ra
            self._rank[ra] += 1

        return True


def kruskal_mst_from_adj_matrix(matrix: List[List[int]]) -> Tuple[List[Tuple[int, int, int]], int]:
    """
    Compute an MST using Kruskal's algorithm from an adjacency matrix.

    matrix[i][j] = weight (> 0) if edge exists, else 0 for no edge.
    Graph assumed undirected (matrix symmetric).

    Returns: (mst_edges, total_weight)
      mst_edges: list of (u, v, w)
    """
    n = len(matrix)

    # Basic validation
    for row in matrix:
        if len(row) != n:
            raise ValueError("Adjacency matrix must be square (n x n).")

    # 1) Extract edges (only upper triangle to avoid duplicates)
    edges: List[Tuple[int, int, int]] = []
    for i in range(n):
        for j in range(i + 1, n):
            w = matrix[i][j]
            if w != 0:
                edges.append((w, i, j))

    # 2) Sort edges by weight
    edges.sort()

    # 3) Kruskal: take smallest edges that don't form a cycle
    dsu = DisjointSet(n)
    mst_edges: List[Tuple[int, int, int]] = []
    total_weight = 0

    for w, u, v in edges:
        if dsu.union(u, v):
            mst_edges.append((u, v, w))
            total_weight += w
            if len(mst_edges) == n - 1:
                break

    # If not enough edges were added, graph wasn't connected
    if n > 0 and len(mst_edges) != n - 1:
        raise ValueError("Graph is not connected; MST does not exist.")

    return mst_edges, total_weight


# --- quick demo ---
if __name__ == "__main__":
    graph = [
        #0 1 2 3 4
        [0, 2, 0, 6, 0],  # 0
        [2, 0, 3, 8, 5],  # 1
        [0, 3, 0, 0, 7],  # 2
        [6, 8, 0, 0, 9],  # 3
        [0, 5, 7, 9, 0],  # 4
    ]

    mst, cost = kruskal_mst_from_adj_matrix(graph)
    print("MST edges:", mst)
    print("Total weight:", cost)


MST edges: [(0, 1, 2), (1, 2, 3), (1, 4, 5), (0, 3, 6)]
Total weight: 16
